# 02 — Pipeline de degradation

Ce notebook prend les plaques nettes generees au notebook 01 et cree des versions degradees (floues, bruitees, compressees).

**Input :** `dataset/clean/` (10K plaques nettes)
**Output :** `dataset/degraded/` (50K images degradees) + `dataset/pairs.json` (mapping floue → nette)

**Runtime :** CPU suffit (~20 min)

In [ ]:
!pip install -q pillow numpy opencv-python-headless matplotlib tqdm

# Charger le dataset depuis Google Drive
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p dataset
!cp -r /content/drive/MyDrive/deblurring_dataset/* dataset/
print("Dataset charge depuis Drive")

In [ ]:
import cv2
import numpy as np
import json
import os
import random
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt

# Charger les metadonnees du notebook 01
with open('dataset/metadata.json') as f:
    metadata = json.load(f)
print(f"{len(metadata)} plaques nettes chargees")

## 1. Fonctions de degradation

In [ ]:
def motion_blur(img, kernel_size=25, angle=0):
    """Motion blur directionnel."""
    kernel = np.zeros((kernel_size, kernel_size))
    kernel[kernel_size // 2, :] = 1.0 / kernel_size
    M = cv2.getRotationMatrix2D((kernel_size / 2, kernel_size / 2), angle, 1)
    kernel = cv2.warpAffine(kernel, M, (kernel_size, kernel_size))
    kernel = kernel / kernel.sum()  # normaliser
    return cv2.filter2D(img, -1, kernel)


def gaussian_blur(img, sigma=5):
    """Blur gaussien."""
    ksize = int(sigma * 6) | 1  # toujours impair
    return cv2.GaussianBlur(img, (ksize, ksize), sigma)


def downsample_upsample(img, factor=4):
    """Simule une plaque loin : reduit puis agrandit."""
    h, w = img.shape[:2]
    small = cv2.resize(img, (w // factor, h // factor), interpolation=cv2.INTER_AREA)
    return cv2.resize(small, (w, h), interpolation=cv2.INTER_LINEAR)


def add_gaussian_noise(img, sigma=25):
    """Bruit gaussien additif."""
    noise = np.random.normal(0, sigma, img.shape).astype(np.float32)
    noisy = np.clip(img.astype(np.float32) + noise, 0, 255)
    return noisy.astype(np.uint8)


def jpeg_compress(img, quality=20):
    """Compression JPEG avec artefacts."""
    _, encoded = cv2.imencode('.jpg', img, [cv2.IMWRITE_JPEG_QUALITY, quality])
    return cv2.imdecode(encoded, cv2.IMREAD_COLOR)


# Degradations composees (combinaisons realistes)
DEGRADATIONS = {
    'motion_light': lambda img: motion_blur(img, kernel_size=random.randint(10, 20), angle=random.uniform(-5, 5)),
    'motion_heavy': lambda img: motion_blur(img, kernel_size=random.randint(25, 45), angle=random.uniform(-15, 15)),
    'gaussian_light': lambda img: gaussian_blur(img, sigma=random.uniform(2, 5)),
    'gaussian_heavy': lambda img: gaussian_blur(img, sigma=random.uniform(6, 12)),
    'lowres': lambda img: downsample_upsample(img, factor=random.choice([3, 4, 6, 8])),
    'noisy': lambda img: add_gaussian_noise(img, sigma=random.uniform(15, 40)),
    'jpeg': lambda img: jpeg_compress(img, quality=random.randint(10, 30)),
    'combo_motion_noise': lambda img: add_gaussian_noise(
        motion_blur(img, random.randint(15, 30), random.uniform(-10, 10)),
        sigma=random.uniform(10, 25)
    ),
    'combo_lowres_jpeg': lambda img: jpeg_compress(
        downsample_upsample(img, factor=random.choice([3, 4, 5])),
        quality=random.randint(15, 40)
    ),
    'combo_all': lambda img: jpeg_compress(
        add_gaussian_noise(
            motion_blur(
                downsample_upsample(img, factor=random.choice([2, 3, 4])),
                kernel_size=random.randint(10, 20),
                angle=random.uniform(-10, 10)
            ),
            sigma=random.uniform(5, 15)
        ),
        quality=random.randint(20, 50)
    ),
}

print(f"{len(DEGRADATIONS)} types de degradation definis")

## 2. Visualiser les degradations

In [ ]:
# Montrer chaque type de degradation sur une plaque
sample_img = cv2.imread('dataset/clean/plate_00000.png')

fig, axes = plt.subplots(2, 5, figsize=(20, 6))
axes_flat = axes.flat

# Original
ax = next(axes_flat)
ax.imshow(cv2.cvtColor(sample_img, cv2.COLOR_BGR2RGB))
ax.set_title('Original (nette)', fontsize=10)
ax.axis('off')

# Chaque degradation
for (name, func), ax in zip(DEGRADATIONS.items(), axes_flat):
    degraded = func(sample_img.copy())
    ax.imshow(cv2.cvtColor(degraded, cv2.COLOR_BGR2RGB))
    ax.set_title(name, fontsize=9)
    ax.axis('off')

plt.suptitle('Types de degradation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Generer le dataset de paires

In [ ]:
os.makedirs('dataset/degraded', exist_ok=True)

DEGRADATIONS_PER_IMAGE = 5  # 5 versions degradees par plaque nette
pairs = []
deg_names = list(DEGRADATIONS.keys())

print(f"Generation de {len(metadata) * DEGRADATIONS_PER_IMAGE} paires...")

for entry in tqdm(metadata):
    clean_path = f"dataset/clean/{entry['filename']}"
    clean_img = cv2.imread(clean_path)

    if clean_img is None:
        continue

    # Choisir 5 degradations aleatoires parmi les 10
    chosen = random.sample(deg_names, DEGRADATIONS_PER_IMAGE)

    for j, deg_name in enumerate(chosen):
        degraded = DEGRADATIONS[deg_name](clean_img.copy())

        deg_filename = f"plate_{entry['id']:05d}_deg{j}_{deg_name}.png"
        cv2.imwrite(f"dataset/degraded/{deg_filename}", degraded)

        pairs.append({
            'clean': entry['filename'],
            'degraded': deg_filename,
            'text': entry['text'],
            'degradation': deg_name,
        })

# Sauvegarder
with open('dataset/pairs.json', 'w') as f:
    json.dump(pairs, f, indent=2)

print(f"\n{len(pairs)} paires generees")
print(f"Sauvegarde dans dataset/pairs.json")

## 4. Split train / val / test

In [ ]:
random.shuffle(pairs)

n = len(pairs)
n_train = int(n * 0.8)
n_val = int(n * 0.1)

splits = {
    'train': pairs[:n_train],
    'val': pairs[n_train:n_train + n_val],
    'test': pairs[n_train + n_val:],
}

for name, data in splits.items():
    with open(f'dataset/{name}.json', 'w') as f:
        json.dump(data, f, indent=2)
    print(f"{name}: {len(data)} paires")

print(f"\nTotal: {n} paires")

In [ ]:
# Verification visuelle : paires aleatoires
fig, axes = plt.subplots(4, 2, figsize=(12, 14))
samples = random.sample(pairs, 4)

for i, p in enumerate(samples):
    clean = cv2.cvtColor(cv2.imread(f"dataset/clean/{p['clean']}"), cv2.COLOR_BGR2RGB)
    deg = cv2.cvtColor(cv2.imread(f"dataset/degraded/{p['degraded']}"), cv2.COLOR_BGR2RGB)

    axes[i][0].imshow(deg)
    axes[i][0].set_title(f"Degradee ({p['degradation']})", fontsize=10)
    axes[i][0].axis('off')

    axes[i][1].imshow(clean)
    axes[i][1].set_title(f"Nette — {p['text']}", fontsize=10)
    axes[i][1].axis('off')

plt.suptitle('Paires degradee → nette (verite terrain)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Archiver le dataset

Pour le transmettre entre notebooks Colab.

In [ ]:
# Sauvegarder sur Google Drive
!cp -r dataset/degraded /content/drive/MyDrive/deblurring_dataset/
!cp dataset/pairs.json dataset/train.json dataset/val.json dataset/test.json /content/drive/MyDrive/deblurring_dataset/
print("Dataset complet sauvegarde sur Drive (clean + degraded + splits)")

## Next

Le dataset de 50K paires est pret. Passe au notebook **03_train_model.ipynb** pour fine-tuner NAFNet.